In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("SalesData").getOrCreate()

# Define the dataset
data = [
    (1, 101, "2024-01", 800),
    (2, 101, "2024-02", 950),
    (3, 101, "2024-03", 700),
    (4, 102, "2024-01", 300),
    (5, 102, "2024-02", 400),
    (6, 102, "2024-03", 600),
    (7, 103, "2024-01", 1200),
    (8, 103, "2024-02", 1100),
    (9, 103, "2024-03", 900),
    (10, 104, "2024-01", 500),
    (11, 104, "2024-02", 550),
    (12, 104, "2024-03", 480)
]

# Define schema
columns = ["id", "product_id", "sale_month", "revenue"]

# Create DataFrame
sales_df = spark.createDataFrame(data, columns)

# Show DataFrame
sales_df.show()


In [0]:
sale_monthly_df=(
    sales_df
    .withColumn("month_rank",
                f.dense_rank()
                .over(Window.partitionBy("product_id")
                      .orderBy(f.desc("sale_month"))
                ))
)
s1_df=sale_monthly_df.filter(f.col("month_rank")==1).alias("s1")
s2_df=sale_monthly_df.filter(f.col("month_rank")==2).alias("s2")

result_df=(
    s1_df.join(s2_df, on="product_id")
    .withColumn("decline_amount", f.col("s2.revenue")-f.col("s1.revenue"))
    .select(
        f.col("s1.product_id"),
        f.col("s1.revenue").alias("current_month_revenue"),
        f.col("s2.revenue").alias("prev_month_revenue"),
        f.col("decline_amount")
    )
    .filter(f.col("decline_amount")>0)
    .orderBy(f.desc("decline_amount"))
)
display(result_df)